In [1]:
import torch.nn.functional as F

In [ ]:
F.leaky_relu()

In [102]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()
import warnings
warnings.filterwarnings('ignore')
import datetime
import json
import re
import gensim
from gensim import corpora, models
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import nltk
# nltk.download('wordnet')
import random
import multiprocessing
from collections import defaultdict
from gensim.models import Word2Vec
from nltk.corpus import stopwords
stopwords = set(stopwords.words('english'))

In [103]:
base_path = '/home/luxiaoling/wukun/MIND/data/'

In [104]:
news_df_path = base_path + 'all_news.tsv'
cols = ['news_id', 'category', 'subcategory', 'topic', 'title_lower', 'abstract_lower', 'clean_title', 'clean_abstract']
news_df = pd.read_csv(news_df_path, sep='\t', names=cols)
news_df.head()

,news_id,category,subcategory,topic,title_lower,abstract_lower,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,163,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N45436,news,newsscienceandtechnology,27,walmart slashes prices on last-generation ipads,apple's new ipad releases bring big deals on l...,walmart slashes prices last generation ipads,apple's new ipad releases bring big deals last...
2,N23144,health,weightloss,175,50 worst habits for belly fat,these seemingly harmless habits are holding yo...,worst habits belly fat,seemingly harmless habits holding back keeping...
3,N86255,health,medical,32,dispose of unwanted prescription drugs during ...,.,dispose unwanted prescription drugs dea's take...,PAD
4,N93187,news,newsworld,104,the cost of trump's aid freeze in the trenches...,lt. ivan molchanets peeked over a parapet of s...,cost trump's aid freeze trenches ukraine's war,lt ivan UNK peeked UNK sand bags front line wa...


In [105]:
news_df.news_id.nunique()

78448

In [106]:
def get_text_len(x):
    try:
        return len(x.split())
    except:
        return 0
    

In [107]:
news_df['title_len'] = news_df['title_lower'].apply(get_text_len)
news_df['abstract_lower'] = news_df['abstract_lower'].apply(get_text_len)

In [109]:
news_df['title_len'].mean()

10.720846930450744

In [110]:
news_df['abstract_lower'].mean()

35.47051550071385

In [65]:
import scipy.sparse as sp

In [66]:
adj = sp.eye(10)
adj

<10x10 sparse matrix of type '<class 'numpy.float64'>'
	with 10 stored elements (1 diagonals) in DIAgonal format>

In [79]:
d_inv = np.power(adj.sum(1), -1).flatten()

In [92]:
d_inv

matrix([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])

In [87]:
sp.diags([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], 0)

<10x10 sparse matrix of type '<class 'numpy.float64'>'
	with 10 stored elements (1 diagonals) in DIAgonal format>

In [100]:
x

tensor([[3., 2.],
        [3., 4.]])

In [101]:
x[:, 1] += torch.as_tensor([5, 6])
x

tensor([[ 3.,  7.],
        [ 3., 10.]])

In [98]:
x[0][0] = 3


In [22]:
idx = tf.where((x > 2) | (x < 2))
tf.gather_nd(x, idx)
# idx

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([1, 3], dtype=int32)>

In [31]:
x = torch.Tensor([[1,2], [3, 4]])

In [28]:
div_term = torch.exp(torch.arange(0, 512, 2) *
                             -(math.log(10000.0) / 512)) # 

In [63]:
def cal_gauc(user_ids, preds, labels):
    assert len(user_ids) == len(preds) and len(preds) == len(labels)
    uid_pred_dict = defaultdict(list)
    uid_label_dict = defaultdict(list)
    for uid, pred, label in zip(user_ids, preds, labels):
        uid_pred_dict[uid].append(pred)
        uid_label_dict[uid].append(label)
    all_auc = 0
    total = 0
    for uid in uid_pred_dict.keys():
        if len(set(uid_label_dict[uid])) < 2:
            continue
        u_auc = metrics.roc_auc_score(uid_label_dict[uid], uid_pred_dict[uid])
        n_u = len(uid_label_dict[uid])
        all_auc += u_auc * n_u
        total += n_u    
    return all_auc / total

In [64]:
user_id = ['a', 'a', 'a', 'b', 'b', 'b', 'a']
label = [1, 0, 1, 1, 1, 1, 0]
pred = [0.4, 0.5, 0.7, 0.2, 0.6, 0.7, 0.4]
group_auc = cal_gauc(user_id, pred, label)
print('group_auc: {:.4f}'.format(group_auc))
auc = metrics.roc_auc_score(label, pred)
print("auc: {:.4f}".format(auc))

group_auc: 0.6250
auc: 0.6500


In [51]:
user_id = ['a', 'a', 'a', 'b', 'b', 'b', 'a']
label = [1, 0, 1, 0, 1, 1, 0]
pred = [0.4, 0.5, 0.7, 0.2, 0.6, 0.7, 0.4]
group_auc = gauc(label, pred, user_id)
print('group_auc: {:.4f}'.format(group_auc))
auc = metrics.roc_auc_score(label, pred)
print("auc: {:.4f}".format(auc))

a 2.5 4
b 5.5 7
group_auc: 0.7857
auc: 0.8750


In [ ]:
msg = 'Test Loss: {0:>5.2}, Test Acc: {1:>6.2%}, Test Auc: {2:>6.4}, new_user Auc: {3:>6.4}, old_user Auc: {4:>6.4}, new_news Auc: {5:>6.4}, old_news Auc: {6:>6.4}, total_loss: {7:>7.4}, rig: {8:>8.4}, gauc: {9:>9.4}'.format(test_loss, test_acc, test_auc, new_user_auc, old_user_auc, new_news_auc, old_news_auc, loss_total, rig, gauc)

In [50]:
import numpy as np
from sklearn.metrics import roc_auc_score
from collections import defaultdict

def gauc(label, pred, user_id):
    '''
    :param label: ground truth
    :param prob: predicted prob
    :param user_id: user index
    :return: gauc
    '''
    if(len(label) != len(user_id)):
        raise ValueError("impression id num should equal to the sample num,"\
                         "impression id num is {}".format(len(user_id)))
    group_truth = defaultdict(lambda: [])
    group_score = defaultdict(lambda: [])
    for idx, truth in enumerate(label):
        uid = user_id[idx]
        group_truth[uid].append(label[idx])
        group_score[uid].append(pred[idx])
    group_flag = defaultdict(lambda: False)
    for uid in set(user_id):
        truths = group_truth[uid]
        for i in range(len(truths)-1):
            if(truths[i] != truths[i+1]):
                flag = True
                break
        group_flag[uid] = flag
    total_auc = 0
    total_impression = 0
    for uid in group_flag:
        if group_flag[uid]:
            total_auc += len(group_truth[uid]) * roc_auc_score(np.asarray(group_truth[uid]), np.asarray(group_score[uid]))
            total_impression += len(group_truth[uid])
            print(uid, total_auc, total_impression)
    group_auc = float(total_auc) / total_impression
    group_auc = round(group_auc, 4)
    return group_auc

In [36]:
F.softmax(x, dim=-1)

tensor([[0.2689, 0.7311],
        [0.2689, 0.7311]])

In [28]:
%%timeit -r 8 -n 200
a = torch.LongTensor([[1, 2, 3], [4, 5, 6]])

4.11 µs ± 2.3 µs per loop (mean ± std. dev. of 8 runs, 200 loops each)


In [39]:
%%timeit -r 1 -n 10
x = np.random.choice(50, 100000)
torch.LongTensor([[0] * y + [1] * (100-y) for y in x])

331 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 10 loops each)


In [41]:
import pandas as pd

In [42]:
df = pd.DataFrame([[1,2], [3,4]], columns=['a', 'b'])

In [43]:
df.head()

,a,b
0,1,2
1,3,4


In [47]:
df.apply(lambda x: x.a + x.b, axis=1)

0    3
1    7
dtype: int64

In [49]:
from collections import defaultdict
import json

In [50]:
d = defaultdict(lambda: defaultdict(int))
d['a']['b'] = 10

In [53]:
ret = {}
for x, y in d.items():
    ret[x] = dict(y)
ret

{'a': {'b': 10}}

In [54]:
with open('test.json', 'w') as f:
    json.dump(ret, f)

In [34]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [40]:
%%timeit -r 1 -n 10
x = np.random.choice(50, 100000)
torch.LongTensor([[0] * y + [1] * (100-y) for y in x]).to(device)

347 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 10 loops each)


In [55]:
from pandarallel import pandarallel  # df.apply加速
# Initialization
pandarallel.initialize(progress_bar=True, nb_workers=16)

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [57]:
df = pd.DataFrame({'a': np.arange(1, 10000), 'b': np.arange(3, 10002)})
df.head()

,a,b
0,1,3
1,2,4
2,3,5
3,4,6
4,5,7


In [58]:
df['a'].parallel_apply(lambda x: x**3)

KeyboardInterrupt: 

Process ForkPoolWorker-107:
Process ForkPoolWorker-106:
Process ForkPoolWorker-102:
Process ForkPoolWorker-109:
Process ForkPoolWorker-110:
Process ForkPoolWorker-108:
Process ForkPoolWorker-103:
Process ForkPoolWorker-100:
Process ForkPoolWorker-101:
Process ForkPoolWorker-104:
Process ForkPoolWorker-112:
Process ForkPoolWorker-111:
Process ForkPoolWorker-105:
Process ForkPoolWorker-114:
Process ForkPoolWorker-113:
Process ForkPoolWorker-99:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.6/multiprocessing/process.py", line 258, in _bootstrap
    self.run()
Traceback (most recent call last):
  F

In [65]:
x = torch.tensor([1,2,3])
y = torch.tensor([1,2,3])
z = torch.tensor([1,2,3])

In [59]:
from torch.utils.data import DataLoader
from prefetch_generator import BackgroundGenerator

In [60]:
dataset = torch.utils.data.TensorDataset(x, y, z)

In [61]:
class DataLoaderX(DataLoader):
    def __iter__(self):
        return BackgroundGenerator(super().__iter__())

In [63]:
DataLoaderX(dataset, 2, shuffle=False, num_worker2=1)

In [38]:
x = np.array([np.exp(-1e7), np.exp(-1e7)])

In [39]:
x / x.sum()

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:1: RuntimeWarning: invalid value encountered in true_divide
  """Entry point for launching an IPython kernel.


array([nan, nan])

In [5]:
a = nn.Embedding(10, 16)
b = nn.Embedding(10, 16)

In [30]:
y = [[5, 6, 7, 8], [1,2, 3, 4]]

In [31]:
[[[0] * x + [1] * (10 - x) for x in z] for z in y]

[[[0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
  [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
  [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
  [0, 0, 0, 0, 0, 0, 0, 0, 1, 1]],
 [[0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
  [0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
  [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
  [0, 0, 0, 0, 1, 1, 1, 1, 1, 1]]]

In [52]:
a, b, c, d, \
e, f = 1,2,3,4,5,6

In [82]:
2232748 / 1024

2180.41796875

In [81]:
5e-3 * 0.08

0.0004

In [94]:
0.985 ** 80

0.29846845658843124

In [91]:
1e-3 * 

1.0000000000000017e-240

In [95]:
cpt_dict = {
    'a': {'b': 1}
}

In [97]:
torch.save(cpt_dict, './test.ckpt')

In [98]:
cpt = torch.load('./test.ckpt')

In [99]:
cpt

{'a': {'b': 1}}

In [100]:
d = {'a': 1, 'b': 2}

In [105]:
np.array(list(d.values())) ** 0.5

array([1.        , 1.41421356])

In [112]:
np.random.choice([1, 2, 3, 4], size=2, replace=False).tolist()

[3, 2]

In [109]:
[3, 4, 5] + np.random.choice([1, 2, 3, 4], size=2, replace=False)

ValueError: operands could not be broadcast together with shapes (3,) (2,) 

In [113]:
from torch.utils.tensorboard import SummaryWriter

In [114]:
writer = SummaryWriter(log_dir='./test_dir')

In [115]:
import random

In [121]:
random.seed(2021)
x = [10, 8, 5, 6, 3, 7]
random.shuffle(x)
x

[8, 7, 10, 5, 3, 6]

In [122]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()
import warnings
warnings.filterwarnings('ignore')
import datetime
from collections import defaultdict

In [123]:
train_behavior_path = '/home/luxiaoling/wukun/MIND/data/large/train/processed_behavior.csv'

In [124]:
train_behavior_df = pd.read_csv(train_behavior_path, sep='\t', names=['uid', 'impr_time', 'history', 'cur_impr'])

In [125]:
train_behavior_df.head()

,uid,impr_time,history,cur_impr
0,U403465,2019-11-09 00:00:00,N59850 N104930 N68866 N82374 N123325 N127916 N...,N92613-0 N17456-0 N67369-0 N31486-0 N76810-0 N...
1,U493092,2019-11-09 00:00:02,N3960 N115631 N21659 N79594 N128643 N52938 N12...,N6865-0 N128031-0 N79990-0 N121426-0 N2198-0 N...
2,U172654,2019-11-09 00:00:03,N43253 N119785 N81937 N128965 N58990 N74795 N7...,N65783-1 N119284-0 N76810-0
3,U248125,2019-11-09 00:00:07,N102417 N22902 N95420 N81970 N24696 N115565 N9...,N90042-0 N2598-0 N108809-0 N104610-0 N18956-0 ...
4,U495159,2019-11-09 00:00:13,N88765 N74317 N63723 N121794 N95711 N64341 N75...,N104437-0 N76810-0 N98657-0 N25492-0 N108809-0...


In [132]:
train_behavior_df['impr_date'] = train_behavior_df.impr_time.apply(lambda x: x.split()[0])
train_behavior_df.head()

,uid,impr_time,history,cur_impr,impr_date
0,U403465,2019-11-09 00:00:00,N59850 N104930 N68866 N82374 N123325 N127916 N...,N92613-0 N17456-0 N67369-0 N31486-0 N76810-0 N...,2019-11-09
1,U493092,2019-11-09 00:00:02,N3960 N115631 N21659 N79594 N128643 N52938 N12...,N6865-0 N128031-0 N79990-0 N121426-0 N2198-0 N...,2019-11-09
2,U172654,2019-11-09 00:00:03,N43253 N119785 N81937 N128965 N58990 N74795 N7...,N65783-1 N119284-0 N76810-0,2019-11-09
3,U248125,2019-11-09 00:00:07,N102417 N22902 N95420 N81970 N24696 N115565 N9...,N90042-0 N2598-0 N108809-0 N104610-0 N18956-0 ...,2019-11-09
4,U495159,2019-11-09 00:00:13,N88765 N74317 N63723 N121794 N95711 N64341 N75...,N104437-0 N76810-0 N98657-0 N25492-0 N108809-0...,2019-11-09


In [127]:
train_behavior_df[train_behavior_df.impr_date == '2019-11-09'].shape

(192552, 5)

In [128]:
train_behavior_df.history.isnull().sum()

46065

In [131]:
train_behavior_df.shape

(2232748, 5)

In [156]:
import datetime, math

In [134]:
datetime_df = train_behavior_df.impr_time.apply(lambda x: datetime.datetime.strptime(x, '%Y-%m-%d %H:%M:%S'))

In [135]:
end_date = datetime.datetime(2019, 11, 15)
train_behavior_df['week'] = datetime_df.apply(lambda x: x.weekday())
train_behavior_df['hour'] = datetime_df.apply(lambda x: x.hour)
interval_df = datetime_df.apply(lambda x: end_date - x)

In [157]:
train_behavior_df['relative_seconds'] = interval_df.apply(lambda x: x.seconds + x.days * 24 * 3600)
train_behavior_df['relative_days'] = train_behavior_df['relative_seconds'].apply(lambda x: math.ceil(x / (24 * 3600)))
train_behavior_df['relative_hours'] = train_behavior_df['relative_seconds'].apply(lambda x: math.ceil(x / 3600))
                                                                                  

In [158]:
train_behavior_df.head()

,uid,impr_time,history,cur_impr,impr_date,week,hour,relative_days,relative_seconds,relative_hours
0,U403465,2019-11-09 00:00:00,N59850 N104930 N68866 N82374 N123325 N127916 N...,N92613-0 N17456-0 N67369-0 N31486-0 N76810-0 N...,2019-11-09,5,0,6,518400,144
1,U493092,2019-11-09 00:00:02,N3960 N115631 N21659 N79594 N128643 N52938 N12...,N6865-0 N128031-0 N79990-0 N121426-0 N2198-0 N...,2019-11-09,5,0,6,518398,144
2,U172654,2019-11-09 00:00:03,N43253 N119785 N81937 N128965 N58990 N74795 N7...,N65783-1 N119284-0 N76810-0,2019-11-09,5,0,6,518397,144
3,U248125,2019-11-09 00:00:07,N102417 N22902 N95420 N81970 N24696 N115565 N9...,N90042-0 N2598-0 N108809-0 N104610-0 N18956-0 ...,2019-11-09,5,0,6,518393,144
4,U495159,2019-11-09 00:00:13,N88765 N74317 N63723 N121794 N95711 N64341 N75...,N104437-0 N76810-0 N98657-0 N25492-0 N108809-0...,2019-11-09,5,0,6,518387,144


In [161]:
train_behavior_df[train_behavior_df.relative_days == 0].head()

,uid,impr_time,history,cur_impr,impr_date,week,hour,relative_days,relative_seconds,relative_hours


In [137]:
val_behavior_path = '/home/luxiaoling/wukun/MIND/data/large/dev/processed_behavior.csv'
val_behavior_df = pd.read_csv(val_behavior_path, sep='\t', names=['uid', 'impr_time', 'history', 'cur_impr'])
val_behavior_df.head()

,uid,impr_time,history,cur_impr
0,U134050,2019-11-15 08:55:22,N12246 N128820 N119226 N4065 N67770 N33446 N10...,N91737-0 N30206-0 N54368-0 N117802-0 N18190-0 ...
1,U254959,2019-11-15 11:42:35,N34011 N9375 N67397 N7936 N118985 N109453 N103...,N119999-0 N24958-0 N104054-0 N33901-0 N9250-0 ...
2,U499841,2019-11-15 09:08:21,N63858 N26834 N6379 N85484 N15229 N65119 N1047...,N18190-0 N89764-0 N91737-0 N54368-0 N49978-1 N...
3,U107107,2019-11-15 05:50:31,N12959 N8085 N18389 N3758 N9740 N90543 N129790...,N122944-1 N18190-0 N55801-0 N59297-0 N128045-0...
4,U492344,2019-11-15 05:02:25,N109183 N48453 N85005 N45706 N98923 N46069 N35...,N64785-0 N82503-0 N32993-0 N122944-0 N29160-0 ...


In [142]:
val_datetime_df = val_behavior_df.impr_time.apply(lambda x: datetime.datetime.strptime(x, '%Y-%m-%d %H:%M:%S'))
end_date = datetime.datetime(2019, 11, 15)
val_behavior_df['week'] = val_datetime_df.apply(lambda x: x.weekday())
val_behavior_df['hour'] = val_datetime_df.apply(lambda x: x.hour)
val_interval_df = val_datetime_df.apply(lambda x: end_date - x)

In [159]:
val_behavior_df['relative_days'] = val_interval_df.apply(lambda x: 0)
val_behavior_df['relative_seconds'] = val_interval_df.apply(lambda x: - x.seconds)
val_behavior_df['relative_hours'] = val_behavior_df['relative_seconds'].apply(lambda x: math.ceil(x / 3600))

In [160]:
val_behavior_df.head()

,uid,impr_time,history,cur_impr,week,hour,relative_days,relative_seconds,relative_hours
0,U134050,2019-11-15 08:55:22,N12246 N128820 N119226 N4065 N67770 N33446 N10...,N91737-0 N30206-0 N54368-0 N117802-0 N18190-0 ...,4,8,0,-54278,-15
1,U254959,2019-11-15 11:42:35,N34011 N9375 N67397 N7936 N118985 N109453 N103...,N119999-0 N24958-0 N104054-0 N33901-0 N9250-0 ...,4,11,0,-44245,-12
2,U499841,2019-11-15 09:08:21,N63858 N26834 N6379 N85484 N15229 N65119 N1047...,N18190-0 N89764-0 N91737-0 N54368-0 N49978-1 N...,4,9,0,-53499,-14
3,U107107,2019-11-15 05:50:31,N12959 N8085 N18389 N3758 N9740 N90543 N129790...,N122944-1 N18190-0 N55801-0 N59297-0 N128045-0...,4,5,0,-65369,-18
4,U492344,2019-11-15 05:02:25,N109183 N48453 N85005 N45706 N98923 N46069 N35...,N64785-0 N82503-0 N32993-0 N122944-0 N29160-0 ...,4,5,0,-68255,-18


In [149]:
val_behavior_df.week.max(), val_behavior_df.hour.max()

(4, 23)

In [162]:
train_behavior_df.week.max(), train_behavior_df.hour.max(), train_behavior_df.relative_days.min()

(6, 23, 1)

In [163]:
train_behavior_df.shape, val_behavior_df.shape

((2232748, 10), (376471, 9))

In [164]:
columns = ['uid', 'impr_time', 'week', 'hour', 'relative_days', 'relative_hours', 'relative_seconds', 'history', 'cur_impr']
train_behavior_df[columns].to_csv(train_behavior_path, sep='\t', header=False, index=False)

val_behavior_df[columns].to_csv(val_behavior_path, sep='\t', header=False, index=False)


In [14]:
import multiprocessing
from tqdm import tqdm

In [21]:
def g(x):
    return 3
def f(x):
    z = g(x)
    return [1,2,[3], z]

with multiprocessing.Pool(16) as pool:
    ret = list(tqdm(pool.imap(f, range(100000)), total=100000))
# ret

100%|██████████| 100000/100000 [00:05<00:00, 18675.71it/s]


In [7]:
def f(x):
    return [1,2,3]

pool = multiprocessing.Pool(processes=4)
# pool.map(f, range(10))
r = pool.map_async(f, range(10))
# DO STUFF
r.wait()

In [8]:
r.get()

[[0], [1], [4], [9], [16], [25], [36], [49], [64], [81]]

In [9]:
import numpy as np

In [10]:
np.concatenate(r.get())

array([ 0,  1,  4,  9, 16, 25, 36, 49, 64, 81])

In [12]:
[x for record in r.get() for x in record]

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

In [13]:
[x for x in zip([1,2,3], [4,5,6])]

[(1, 4), (2, 5), (3, 6)]